# 🚀 Colab AI Studio — Worker Distant GPU (Tesla T4)

Ce notebook transforme cette session Google Colab en worker de calcul distant pour **Colab AI Studio**.

**Prérequis** : Assurez-vous d'avoir activé l'accélération GPU : `Exécution > Modifier le type d'exécution > T4 GPU`.

In [ ]:
# [1/4] Vérification du GPU NVIDIA Tesla T4
!nvidia-smi

In [ ]:
# [2/4] Installation des dépendances du worker, des pipelines et du tunnel
!pip install -q fastapi uvicorn pydantic pycloudflared pyngrok nest_asyncio httpx \
  diffusers transformers accelerate

In [ ]:
# [3/4] Clonage du dépôt et initialisation de l'automate ModelSwapper
!git clone https://github.com/personnal-agent/colab-ai-studio.git /content/colab-ai-studio \
  2>/dev/null || (cd /content/colab-ai-studio && git pull)

import os
import secrets
import sys

if '/content/colab-ai-studio' not in sys.path:
    sys.path.insert(0, '/content/colab-ai-studio')

from app.colab_worker.model_swapper import ModelSwapper
from app.colab_worker.server import create_worker_app

# Mode réel ou simulation (par défaut simulation pour test immédiat sans télécharger 20 Go)
SIMULATION_MODE = os.environ.get("COLAB_STUDIO_SIMULATION", "1") == "1"

def load_flux_real(mid: str):
    import torch
    from diffusers import FluxPipeline
    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16
    )
    return pipe.to("cuda")

def load_llama_real(mid: str):
    from transformers import pipeline
    return pipeline(
        "text-generation", model="meta-llama/Meta-Llama-3-8B-Instruct", device_map="auto"
    )

def load_video_real(mid: str):
    import torch
    if mid == "wan-2.1-t2v":
        from diffusers import AutoPipelineForText2Video
        return AutoPipelineForText2Video.from_pretrained(
            "Wan-AI/Wan2.1-T2V-1.3B-Diffusers", torch_dtype=torch.bfloat16
        ).to("cuda")
    from diffusers import CogVideoXPipeline
    return CogVideoXPipeline.from_pretrained(
        "THUDM/CogVideoX-5b", torch_dtype=torch.float16
    ).to("cuda")

loaders = (
    {
        "flux-schnell": load_flux_real,
        "llama-3-8b-instruct": load_llama_real,
        "wan-2.1-t2v": load_video_real,
        "cogvideox-5b": load_video_real,
    }
    if not SIMULATION_MODE else {}
)

WORKER_SECRET = secrets.token_hex(16)
swapper = ModelSwapper(idle_timeout_seconds=300.0)
app = create_worker_app(
    model_swapper=swapper,
    worker_secret=WORKER_SECRET,
    model_loaders=loaders,
    simulation_mode=SIMULATION_MODE,
)

In [ ]:
# [4/4] Ouverture du tunnel Cloudflare et lancement du serveur uvicorn
import nest_asyncio
import uvicorn
from pycloudflared import try_cloudflare

nest_asyncio.apply()
PORT = 8000
tunnel = try_cloudflare(port=PORT)
tunnel_url = tunnel.tunnel.rstrip('/')
print(f"\n=== 🚀 COLAB WORKER : {tunnel_url} | 🔑 SECRET : {WORKER_SECRET} ===\n")
uvicorn.run(app, host="127.0.0.1", port=PORT)